In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [6]:
df = pd.read_csv('carteiras.csv').drop(columns=['Unnamed: 0'])

In [7]:
df

,ano,ativo,peso
0,2016,ANIM3,0.0358
1,2016,AXIA3,0.0770
2,2016,BRAP4,0.0465
3,2016,CSMG3,0.1468
4,2016,ENGI11,0.1797
...,...,...,...
103,2026,SAPR11,0.0310
104,2026,TEND3,0.0236
105,2026,VALE3,0.1367
106,2026,VBBR3,0.0434


In [15]:
pd.read_csv('retornos_ibov/df_ibov_2016.csv').set_index('Date')

,IBOV
Date,
2016-04-01,0.010129
2016-04-04,-0.035244
2016-04-05,0.005617
2016-04-06,-0.019529
2016-04-07,0.008670
...,...
2017-03-27,0.007110
2017-03-28,0.005163
2017-03-29,0.013738


### ANO IGUAL


In [16]:
anos = df.ano.unique()
print(f"ANOS: {anos}")
for ano in anos:
    df_ano = df[df['ano']==ano]
    pesos = df_ano.set_index('ativo')['peso']
    # print(df_ano)
    lista_ativos = df_ano.ativo.unique().tolist()
    print(f"Ativos do ano {ano}: {lista_ativos}")
    # essa lista de ativos é da otimização do ANO X com score do ano x-1
    # Testar essa lista de ativos do ano x para o ano x +1
    ano_teste = ano
    print(f"Carteira do ano {ano} testada para mesmo ano {ano_teste}")
    nome_df = f'retornos_ativos/df_ativos_{ano_teste}.csv'

    nome_ibov = f'retornos_ibov/df_ibov_{ano_teste}.csv'
    try:
        df_retorno = pd.read_csv(nome_df).set_index('date')
        df_retorno = df_retorno[lista_ativos]
        # print(pesos)
        # print(df_retorno)
        carteira_otimizada = (df_retorno*pesos).sum(axis=1)
        carteira_otimizada1 = (carteira_otimizada+1)
        acumulado = carteira_otimizada1.cumprod()*100
        # print(acumulado)
        acumulado.to_csv(f'plotagem_streamlit/acumulado/acumulado_{ano_teste}.csv')

        df_ibov = pd.read_csv(nome_ibov).set_index('Date')
        df_ibov = df_ibov['IBOV']
        ibov_acumulado = (df_ibov+1).cumprod()*100
        ibov_acumulado.to_csv(f'plotagem_streamlit/acumulado_ibov/ibov_acumulado_{ano_teste}.csv')
        
    except Exception as e:
        pass

ANOS: [2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026]
Ativos do ano 2016: ['ANIM3', 'AXIA3', 'BRAP4', 'CSMG3', 'ENGI11', 'FLRY3', 'MGLU3', 'MOVI3', 'PRIO3', 'SUZB3']
Carteira do ano 2016 testada para mesmo ano 2016
Ativos do ano 2017: ['ABEV3', 'ANIM3', 'CVCB3', 'ENGI11', 'IRBR3', 'PSSA3', 'RENT3', 'SLCE3', 'SUZB3', 'VALE3']
Carteira do ano 2017 testada para mesmo ano 2017
Ativos do ano 2018: ['BPAC11', 'CXSE3', 'DIRR3', 'ENEV3', 'IRBR3', 'ISAE4', 'MDNE3', 'PRIO3', 'SUZB3', 'TOTS3']
Carteira do ano 2018 testada para mesmo ano 2018
Ativos do ano 2019: ['CPLE3', 'CXSE3', 'ENEV3', 'IGTI11', 'JHSF3', 'MGLU3', 'RADL3', 'SMFT3', 'VBBR3', 'WEGE3']
Carteira do ano 2019 testada para mesmo ano 2019
Ativos do ano 2020: ['BPAC11', 'CSNA3', 'ENEV3', 'MGLU3', 'PRIO3', 'SLCE3', 'SUZB3', 'TAEE11', 'VALE3', 'VBBR3']
Carteira do ano 2020 testada para mesmo ano 2020
Ativos do ano 2021: ['ALOS3', 'BEEF3', 'BRAP4', 'CPFE3', 'PETR3', 'PETR4', 'SMTO3', 'TAEE11', 'TOTS3', 'VIVT3']
Carteira do ano 202

### ANO PRA FRENTE


In [17]:
anos = df.ano.unique()
print(f"ANOS: {anos}")
for ano in anos:
    df_ano = df[df['ano']==ano]
    pesos = df_ano.set_index('ativo')['peso']
    # print(df_ano)
    lista_ativos = df_ano.ativo.unique().tolist()
    # print(f"Ativos do ano {ano}: {lista_ativos}")
    # essa lista de ativos é da otimização do ANO X com score do ano x-1
    # Testar essa lista de ativos do ano x para o ano x +1
    ano_teste = ano+1
    # print(f"Carteira do ano {ano} testada para ano a frente {ano_teste}")
    nome_df = f'retornos_ativos/df_ativos_{ano_teste}.csv'
    nome_ibov = f'retornos_ibov/df_ibov_{ano_teste}.csv'

    try:
        df_retorno = pd.read_csv(nome_df).set_index('date')
        df_retorno = df_retorno[lista_ativos]
        # print(pesos)
        # print(df_retorno)
        carteira_otimizada = (df_retorno*pesos).sum(axis=1)
        carteira_otimizada1 = (carteira_otimizada+1)
        acumulado = carteira_otimizada1.cumprod()*100
        # print(acumulado)
        acumulado.to_csv(f'plotagem_streamlit/acumulado_f/acumulado_{ano_teste}.csv')

        df_ibov = pd.read_csv(nome_ibov).set_index('Date')
        df_ibov = df_ibov['IBOV']
        ibov_acumulado = (df_ibov+1).cumprod()*100
        ibov_acumulado.to_csv(f'plotagem_streamlit/acumulado_ibov_f/ibov_acumulado_{ano_teste}.csv')
    except Exception as e:
        pass

ANOS: [2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026]
